# H Adatom Intradimer Diffusion Barrier on Si(100)

> **U. Hansen, P. Vogl**
> "Hydrogen passivation of silicon surfaces: A classical molecular-dynamics study"
> Physical Review B 57, 13295 (1998)
> [DOI: 10.1103/PhysRevB.57.13295](https://doi.org/10.1103/PhysRevB.57.13295)

Computes the diffusion barrier of an H adatom on Si(100)2×1 from the materials created in the
[structure notebook](defect_point_adatom_silicon_hydrogen.ipynb): the start, with the H on a dimer atom, and the end of
each path are relaxed with PBE on the platform, and a climbing-image nudged elastic band (NEB) runs from the relaxed
start to each relaxed end. The barrier, from the start and from the end, is compared with Hansen & Vogl's Table V, 1.1 eV
for the intradimer path (a), and with the LDA and GGA values the table quotes; the dimer bond at the saddle with their
3.23 Å.

- The manuscript minimises the energy at a grid of lateral H positions and reads the maximum along each path; the climbing-image NEB finds the saddle along one path directly.
- The manuscript uses a classical Si–H potential with symmetric dimers; the slab here has buckled dimers, as the first-principles results it quotes.
- All atoms are free, as in Table V's relaxed rows.
- Si is at Standata's lattice constant, 5.469 Å (the manuscript states none). The slab has 64 Si (manuscript: 144 Si); the structure notebook sets its size.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](defect_point_adatom_silicon_hydrogen.ipynb), which saves `H on Si(100) start` and `H on Si(100) intradimer end` to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.4 (or use the defaults); each path in `END_NAME_BY_PATH` gets its own relaxation of the end and its own NEB.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the calculations to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the start and the ends by name and print their atom counts.
1. Configure the shared DFT model: the DFT model of the relaxations and the NEB.
1. Configure compute: create a compute configuration on the cluster of `CLUSTER_NAME`, or on the first listed cluster if it is not set.
1. Relax the start and the ends on the platform.
1. Run one climbing-image NEB per path between the relaxed start and the relaxed end.
1. Retrieve the results: the barriers, the largest image error and the two Si nearest the H at the climbing image.
1. Compare with Hansen & Vogl (1998), Table V, and plot the energy profile.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Materials

In [ ]:
# Names saved by defect_point_adatom_silicon_hydrogen.ipynb: the start, and the end of each path to run (Fig. 1)
START_NAME = "H on Si(100) start"
END_NAME_BY_PATH = {
    "intradimer": "H on Si(100) intradimer end",
    # "intrarow": "H on Si(100) intrarow end",
    # "inter-row": "H on Si(100) inter-row end",
}

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
NEB_WORKFLOW_SEARCH_TERM = "neb.json"
APPLICATION_NAME = "espresso"
APPLICATION_VERSION = "7.5"

# full host name of the cluster
CLUSTER_NAME = None
QUEUE_NAME = QueueName.OF
NODES = 1
PPN = 60
TIME_LIMIT_HOURS = 12
TIME_LIMIT = f"{TIME_LIMIT_HOURS:02d}:00:00"
# pw.x and neb.x stop 10 minutes before TIME_LIMIT and write their files
MAX_SECONDS = TIME_LIMIT_HOURS * 3600 - 600

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.4. DFT model parameters

In [ ]:
# GBRV ultrasoft pseudopotentials at their converged 40/200 Ry
FUNCTIONAL = "pbe"
PSEUDOPOTENTIAL_TYPE = "us"
ECUTWFC = 40   # Ry
ECUTRHO = 200  # Ry
# for the 64-Si slab; [1, 2, 1] for the manuscript's 144-Si slab
KGRID = [2, 3, 1]
# &SYSTEM of the relaxations and the NEB: spin-polarized, total magnetization 1
QE_SYSTEM_SETTINGS = {
    "degauss": 0.005,
    "nspin": 2,
    "tot_magnetization": 1,
    "starting_magnetization": 0.5,
}
# &CONTROL of the relaxations: all atoms free, until every force component is below QE's default 1e-3 Ry/Bohr
# (manuscript: 1e-4 eV/Å = 3.9e-6 Ry/Bohr)
RELAXATION_SETTINGS = {"forc_conv_thr": 1e-3, "max_seconds": MAX_SECONDS}
# images between the end points; the path is converged when the largest image error is below 0.1 eV/Å
N_IMAGES = 1
# neb.x stops after nstep_path steps, converged or not; the largest image error is printed in 9
NEB_PATH_SETTINGS = {"nstep_path": 50, "minimum_image": True}

RELAX_UNIT = "pw_relax"
NEB_UNIT = "neb"
# written by neb.x: every image of the final path
NEB_PATH_FILE = "__prefix__.xyz"
# written by neb.x: reaction coordinate, energy and error of every image of the final path
NEB_ENERGY_FILE = "__prefix__.dat"

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load the start and the ends from the uploads folder or the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import load_material

material_names = [START_NAME, *END_NAME_BY_PATH.values()]
materials_by_name = {name: load_material(client, FOLDER, name, ACCOUNT_ID) for name in material_names}
for name, material in materials_by_name.items():
    print(f"{name}: {material.basis.number_of_atoms} atoms")

## 4. Configure the shared DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
if not CLUSTER_NAME:
    clusters = client.clusters.list()
    print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

cluster = {"hostname": CLUSTER_NAME, "fqdn": CLUSTER_NAME} if CLUSTER_NAME else clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, nodes=NODES, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, nodes: {NODES}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Relaxation of the start and the ends

Each material, as built in the structure notebook, is relaxed with the settings of 1.4 on the platform; the relaxed structures are the end points of the NEB.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid, patch_workflow_qe_input
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.core.entity.material.api import find_relaxed_material, get_or_create_material

relaxation_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    RELAX_WORKFLOW_SEARCH_TERM
)
relaxed_materials = {}
relax_job_ids = {}
new_job_ids = []
for name, material in materials_by_name.items():
    saved_material = get_or_create_material(client, material, ACCOUNT_ID)
    relaxed_materials[name] = find_relaxed_material(client, material, ACCOUNT_ID, kgrid=KGRID, unit_name=RELAX_UNIT)
    if relaxed_materials[name] is not None:
        continue
    workflow = Workflow.create(relaxation_workflow_config)
    workflow.name = "Fixed-cell Relaxation H on Si(100)"
    for subworkflow in workflow.subworkflows:
        for application in (subworkflow.application, *(unit.application for unit in subworkflow.units)):
            application.version = APPLICATION_VERSION
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=RELAX_UNIT)
    apply_scf_kgrid(workflow, KGRID, material=material, unit_name=RELAX_UNIT)
    patch_workflow_qe_input(workflow, {"control": RELAXATION_SETTINGS, "system": QE_SYSTEM_SETTINGS}, [RELAX_UNIT])
    job = find_job_for_material(client, saved_material["_id"], workflow.name, ACCOUNT_ID,
                                statuses=("submitted", "queued", "active", "finished"),
                                kgrid=KGRID, unit_name=RELAX_UNIT)
    if job is None:
        job = create_job(
            api_client=client, materials=[saved_material], workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"Fixed-cell Relaxation {name} {timestamp}",
        )
        new_job_ids.append(job["_id"])
    relax_job_ids[name] = job["_id"]

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async
from mat3ra.notebooks_utils.core.entity.material.api import get_final_structure_for_job

submit_jobs(client.jobs, new_job_ids)
if relax_job_ids:
    await wait_for_jobs_to_finish_async(client.jobs, list(relax_job_ids.values()), poll_interval=POLL_INTERVAL)
for name, job_id in relax_job_ids.items():
    relaxed_materials[name] = get_final_structure_for_job(client, job_id)

## 7. Nudged elastic band per path

The relaxed start is the first image and the relaxed end the last; neb.x interpolates `N_IMAGES` images between them and climbs the highest one to the saddle point. Both list their atoms in the same order, so each atom moves to its own position.

In [ ]:
neb_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    NEB_WORKFLOW_SEARCH_TERM
)
neb_job_ids = {}
for path, end_name in END_NAME_BY_PATH.items():
    path_materials = [relaxed_materials[START_NAME], relaxed_materials[end_name]]
    workflow = Workflow.create(neb_workflow_config)
    workflow.name = f"Nudged Elastic Band H on Si(100) {path}"
    for subworkflow in workflow.subworkflows:
        for application in (subworkflow.application, *(unit.application for unit in subworkflow.units)):
            application.version = APPLICATION_VERSION
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=NEB_UNIT)
    apply_scf_kgrid(workflow, KGRID, material=path_materials[0], unit_name=NEB_UNIT)
    # nowf: neb.x writes no wavefunction files; each image's SCF starts from atomic wavefunctions
    patch_workflow_qe_input(workflow, {"control": {"max_seconds": MAX_SECONDS, "disk_io": "nowf"},
                                       "system": QE_SYSTEM_SETTINGS, "path": NEB_PATH_SETTINGS}, [NEB_UNIT])
    neb_unit = workflow.subworkflows[0].get_unit_by_name(name=NEB_UNIT)
    neb_unit.add_context({"name": "neb", "isEdited": True, "data": {"nImages": N_IMAGES}, "extraData": {}})
    job = find_job_for_material(client, path_materials[0].id, workflow.name, ACCOUNT_ID,
                                statuses=("submitted", "queued", "active", "finished"),
                                kgrid=KGRID, unit_name=NEB_UNIT)
    if job is None:
        materials_set_name = f"H on Si(100) {path} {path_materials[0].id}-{path_materials[1].id}"
        materials_set = client.materials.create_set({"name": materials_set_name, "owner": {"_id": ACCOUNT_ID}})
        for material in path_materials:
            client.materials.move_to_set(material.id, "", materials_set["_id"])
        job = create_job(
            api_client=client, materials=path_materials, workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
            materials_set=materials_set,
        )
        submit_jobs(client.jobs, [job["_id"]])
        print(f"✅ {path}: created NEB job {job['_id']}")
    else:
        print(f"♻️  {path}: reusing existing NEB job {job['_id']}")
    neb_job_ids[path] = job["_id"]

In [ ]:
await wait_for_jobs_to_finish_async(client.jobs, list(neb_job_ids.values()), poll_interval=POLL_INTERVAL)

## 8. Retrieve the results
### 8.1. Energy profile, barriers, the largest image error and the two Si nearest the H at the climbing image

neb.x writes the energy of each image relative to the first one, the start, against the reaction coordinate from 0 to 1: the barrier from the start is the highest, the barrier from the end the highest minus the last image's.

In [ ]:
from io import StringIO
from ase.io import read
from mat3ra.notebooks_utils.io import read_from_url

barriers_from_start, barriers_from_end, largest_image_errors = {}, {}, {}
reaction_coordinates, image_energies = {}, {}
nearest_silicon_distances, nearest_silicon_separations = {}, {}
for path, job_id in neb_job_ids.items():
    files = client.jobs.list_files(job_id)
    path_file = next(file for file in files if file["key"].endswith(NEB_PATH_FILE))
    energy_file = next(file for file in files if file["key"].endswith(NEB_ENERGY_FILE))
    images = read(StringIO(await read_from_url(path_file["signedUrl"])), index=":", format="xyz")
    energy_table = await read_from_url(energy_file["signedUrl"])
    largest_image_errors[path] = max(float(line.split()[2]) for line in energy_table.splitlines()[1:-1])
    reaction_coordinates[path] = [float(line.split()[0]) for line in energy_table.splitlines()]
    energies = [float(line.split()[1]) for line in energy_table.splitlines()]
    image_energies[path] = energies
    barriers_from_start[path] = max(energies)
    barriers_from_end[path] = max(energies) - energies[-1]
    climbing_image = images[energies.index(max(energies))]
    climbing_image.set_cell(materials_by_name[START_NAME].lattice.vector_arrays)
    climbing_image.pbc = True
    hydrogen_index = climbing_image.symbols.search("H")[0]
    silicon_indices = climbing_image.symbols.search("Si")
    distances = climbing_image.get_distances(hydrogen_index, silicon_indices, mic=True)
    nearest = distances.argsort()[:2]
    nearest_silicon_distances[path] = distances[nearest]
    nearest_silicon_separations[path] = climbing_image.get_distance(*silicon_indices[nearest], mic=True)

## 9. Comparison with Hansen & Vogl (1998), Table V

In [ ]:
# Hansen & Vogl 1998, Table V, relaxed Si, in eV: their barrier and the LDA and GGA barriers they quote
TABLE_V_BY_PATH = {
    "intradimer": {"hansen_vogl": 1.1, "lda": 1.2, "gga": 1.9},
    "intrarow": {"hansen_vogl": 1.8, "lda": 1.3, "gga": 2.4},
    "inter-row": {"hansen_vogl": 2.4, "lda": 2.2, "gga": 3.1},
}
# Hansen & Vogl 1998, p. 13298: the dimer bond at the relaxed intradimer saddle, in Å
DIMER_BOND_AT_SADDLE_BY_PATH = {"intradimer": 3.23}

for path in END_NAME_BY_PATH:
    table_v = TABLE_V_BY_PATH[path]
    print(f"{path.capitalize()} barrier, relaxed, from the start: {barriers_from_start[path]:.2f} eV, from the end: "
          f"{barriers_from_end[path]:.2f} eV, Hansen & Vogl 1998: {table_v['hansen_vogl']} eV; "
          f"Vittadini et al. 1993 (LDA) {table_v['lda']}, Nachtigall & Jordan 1995 (GGA) {table_v['gga']}, "
          f"via Hansen & Vogl Table V")
    print(f"Largest image error, {path} path: {largest_image_errors[path]:.3f} eV/Å")
    print(f"Two Si nearest the H at the climbing image, {path} path: {nearest_silicon_distances[path][0]:.2f} Å and "
          f"{nearest_silicon_distances[path][1]:.2f} Å")
    if path in DIMER_BOND_AT_SADDLE_BY_PATH:
        print(f"Dimer bond at the saddle, H on Si(100): {nearest_silicon_separations[path]:.2f} Å, "
              f"Hansen & Vogl 1998: {DIMER_BOND_AT_SADDLE_BY_PATH[path]} Å")

In [ ]:
from matplotlib import pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

figure, axes = plt.subplots()
for path in END_NAME_BY_PATH:
    highest_image_index = image_energies[path].index(barriers_from_start[path])
    manuscript_barrier = TABLE_V_BY_PATH[path]["hansen_vogl"]
    axes.plot(reaction_coordinates[path], image_energies[path], marker="o",
              label=f"ours, {path}, barrier {barriers_from_start[path]:.2f}")
    axes.plot(reaction_coordinates[path][highest_image_index], manuscript_barrier, marker="s", linestyle="none",
              color="gray", label=f"Hansen & Vogl 1998, {path}, barrier {manuscript_barrier}")
axes.set_xlabel("Reaction coordinate")
axes.set_ylabel("E − E(start) (eV)")
axes.set_title("H adatom diffusion on Si(100)")
axes.legend()
display_matplotlib_figure(figure)

## References

[1] U. Hansen and P. Vogl, "Hydrogen passivation of silicon surfaces: A classical molecular-dynamics study", Phys. Rev. B 57, 13295 (1998). [DOI: 10.1103/PhysRevB.57.13295](https://doi.org/10.1103/PhysRevB.57.13295)

[2] K. F. Garrity, J. W. Bennett, K. M. Rabe and D. Vanderbilt, "Pseudopotentials for high-throughput DFT calculations", Comput. Mater. Sci. 81, 446 (2014). https://doi.org/10.1016/j.commatsci.2013.08.053

[3] A. Vittadini, A. Selloni and M. Casarin, "Energetics of atomic hydrogen diffusion on Si(100)", Surf. Sci. Lett. 289, L625 (1993). [DOI: 10.1016/0167-2584(93)90751-4](https://doi.org/10.1016/0167-2584(93)90751-4)

[4] P. Nachtigall and K. D. Jordan, "Barriers for hydrogen atom diffusion on the Si(100)-2×1 surface", J. Chem. Phys. 102, 8249 (1995). [DOI: 10.1063/1.468954](https://doi.org/10.1063/1.468954)